**Fashion MNIST - A Multilayer Perceptron**

In this practical exercise, we'll make a simple neural network that gets ~90% accuracy on the Fashion MNIST dataset (a ten class, 28x28 image classification problem).

For details about the dataset, check the following link:
**Resources:**
[Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist)


The library we are going to use is Keras to implement the MLP. Check the following link for documentation:
[Keras](https://keras.io)

### Importing The Packages

First up: importing modules. This model just feeds forwards, so we can use a `Sequential` class. As for the layers themselves, we're only using `Dense` and `Activation`. Nothing fancy.

In [1]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.utils import to_categorical
import tensorflow.keras as tf

import pandas as pd
import numpy as np

### Data Loading and Prepartion

Next some constants. `INPUT_SHAPE` is 784 (28 x 28 - flattened form of the image), and `NUM_CATEGORIES` is 10. All fairly self explanatory.  At the bottom, we use `pd.read_csv` to pull in our data, and we grab the `values` property, which is a numpy array version of the `DataFrame` we just read in.


#### Note: Since the files of the data are large, you need to download and unzip it locally from: https://www.kaggle.com/datasets/zalando-research/fashionmnist/data

In [7]:
#  DEFINE CONSTANTS

INPUT_SHAPE = 784
NUM_CATEGORIES = 10

LABEL_DICT = {
 0: "T-shirt/top",
 1: "Trouser",
 2: "Pullover",
 3: "Dress",
 4: "Coat",
 5: "Sandal",
 6: "Shirt",
 7: "Sneaker",
 8: "Bag",
 9: "Ankle boot"
}

# LOAD THE RAW DATA
train_raw = pd.read_csv('./archive/fashion-mnist_train.csv').values
test_raw = pd.read_csv('./archive/fashion-mnist_test.csv').values

Next, we split the import data into training and testing data (as well as X and Y). Any "x" variable is an input, while "y" is the expected output. We set train and test x to everything but the first column of data in our input data (hence the slice), and use Keras' `to_categorical` to one-hot encode the output label to a vector of length `NUM_CATEGORIES` (10). We then normalize the X data. We change the range from 0 - 255 to 0 - 1 by dividing by 255

In [8]:
# split into X and Y, after one-hot encoding
train_x, train_y = (train_raw[:,1:], to_categorical(train_raw[:,0], num_classes = NUM_CATEGORIES))
test_x, test_y = (test_raw[:,1:], to_categorical(test_raw[:,0], num_classes = NUM_CATEGORIES))

# normalize the x data
train_x = train_x / 255
test_x = test_x / 255

### Model Creation

#### Creating the model

Now for the fun part - defining our model! In this case it's a simple four layer network - an input shape of `INPUT_SHAPE` (784), three 512 neuron layers, and an output layer with `NUM_CATEGORIES` neurons (10). We use categorical crossentroy as our loss, as we've got a multi-class classification problem. For an activation function, we use ReLU all the way, except for the output layer, which uses softmax.

In [11]:
# BUILD THE MODEL
model = Sequential()

# Add a Dense layer to the model with 512 neurons
model.add(Dense(512, input_shape=(INPUT_SHAPE,)))

# Add a relu activation
model.add(Activation('relu'))


# Repeat the same process two more times: 
# Dense of 512 followed by relu activation + 
# Dense of 512 followed by relu activation +
# Dense of 512 followed by relu activation

model.add(Dense(512))
model.add(Activation('relu'))


model.add(Dense(512))
model.add(Activation('relu'))

model.add(Dense(NUM_CATEGORIES))
#In the last layer, we add NUM_CATEGORIE as the output size. We still need to add a softmax activation!
model.add(Activation('softmax'))

#### Compiling the model - categorical crossentropy is for multiple choice classification

In [12]:
# Compile the model. Use rmsprop as optimizer, categorical_crossentropy as loss, and accuracy as metrics
# You can check the Keras documentation for samples

model.compile(
    optimizer='rmsprop',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

#### Training the model

Finally, the training. We tell it to use our `train_x` and `train_y` as our training data, `test_x` and `test_y` to validate, use 32 samples per training pass, and run through the whole dataset 8 times.

In [13]:
# train the model!
model.fit(train_x,
          train_y,
          epochs = 8,
          batch_size = 32,
          validation_data = (test_x, test_y))

Epoch 1/8
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.8114 - loss: 0.5234 - val_accuracy: 0.8495 - val_loss: 0.4253
Epoch 2/8
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.8543 - loss: 0.4175 - val_accuracy: 0.8612 - val_loss: 0.4436
Epoch 3/8
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.8624 - loss: 0.3975 - val_accuracy: 0.8580 - val_loss: 0.4454
Epoch 4/8
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.8688 - loss: 0.3860 - val_accuracy: 0.8631 - val_loss: 0.4367
Epoch 5/8
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.8714 - loss: 0.3806 - val_accuracy: 0.8683 - val_loss: 0.4155
Epoch 6/8
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.8743 - loss: 0.3753 - val_accuracy: 0.8532 - val_loss: 0.4068
Epoch 7/8
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.8749 - loss: 0.3734 - val_accuracy: 0.8689 - val_loss: 0.4666
Epoch 8/8
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.8754 - loss: 0.3787 - 

In [14]:
# how'd the model do?
# Perform validation on the model for the train and test data

# Evaluate on train data
train_loss, train_acc = model.evaluate(train_x, train_y)

# Evaluate on test data
test_loss, test_acc = model.evaluate(test_x, test_y)

print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8879 - loss: 0.3425
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8739 - loss: 0.4398
Train accuracy: 0.8878666758537292
Test accuracy: 0.8738999962806702


Nice! The first parameter is loss, while the second parameter is accuracy. almost 90%.